# MRPC Train Set Preprocessing
## Load MSR Paraphrase training data

In [2]:
import pandas as pd

df = pd.read_csv(
    "msr_paraphrase_train.txt",
    sep="\t",
    on_bad_lines="skip"
)

print(df.head())

   Quality    #1 ID    #2 ID  \
0        1   702876   702977   
1        0  2108705  2108831   
2        1  1330381  1330521   
3        0  3344667  3344648   
4        1  1236820  1236712   

                                           #1 String  \
0  Amrozi accused his brother, whom he called "th...   
1  Yucaipa owned Dominick's before selling the ch...   
2  They had published an advertisement on the Int...   
3  Around 0335 GMT, Tab shares were up 19 cents, ...   
4  The stock rose $2.11, or about 11 percent, to ...   

                                           #2 String  
0  Referring to him as only "the witness", Amrozi...  
1  Yucaipa bought Dominick's in 1995 for $693 mil...  
2  On June 10, the ship's owners had published an...  
3  Tab shares jumped 20 cents, or 4.6%, to set a ...  
4  PG&E Corp. shares jumped $1.63 or 8 percent to...  


## Inspect column names

In [4]:
df.columns

Index(['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'], dtype='object')

## Show DataFrame info

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3938 entries, 0 to 3937
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Quality    3938 non-null   int64 
 1   #1 ID      3938 non-null   int64 
 2   #2 ID      3938 non-null   int64 
 3   #1 String  3938 non-null   object
 4   #2 String  3917 non-null   object
dtypes: int64(3), object(2)
memory usage: 154.0+ KB


## Drop ID columns

In [6]:
df = df.drop(columns=["#1 ID", "#2 ID"])

## Drop rows with missing strings and reset index

In [7]:
df = df.dropna(subset=["#1 String", "#2 String"])
df = df.reset_index(drop=True)


## Rename columns to text1, text2, label

In [8]:
df = df.rename(columns={
    "Quality": "label",
    "#1 String": "text1",
    "#2 String": "text2"
})

df = df[["text1", "text2", "label"]]

## Show cleaned DataFrame info

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3917 entries, 0 to 3916
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text1   3917 non-null   object
 1   text2   3917 non-null   object
 2   label   3917 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 91.9+ KB


## Cast text columns to string type

In [10]:
df["text1"] = df["text1"].astype('string')
df["text2"] = df["text2"].astype('string')

## Define and apply whitespace cleaning function

In [ ]:
import re

def clean_text(text):
    text = str(text)
    
    text = re.sub(r'\s+', ' ', text)
   
    text = text.strip()
    
    return text

df["text1"] = df["text1"].apply(clean_text)
df["text2"] = df["text2"].apply(clean_text)

## Filter short texts and remove duplicates

In [12]:
df = df[df["text1"].str.len() > 10]
df = df[df["text2"].str.len() > 10]

df = df.drop_duplicates()

df = df.reset_index(drop=True)

## Show final shape

In [13]:
df.shape

(3916, 3)

## Check label distribution

In [14]:
print(df["label"].value_counts())

label
1    2646
0    1270
Name: count, dtype: int64


## Save cleaned training data to CSV

In [15]:
df.to_csv("mrpc_train_clean.csv", index=False)